#Часть 3. Демо (3 балла)
В данной части вам необходимо реализовать демо-приложение на Gradio для моделей, полученных в предыдущих частях.

**Требования**:

- Демо должно поддерживать обе задачи (классификацию и детекцию), пользователь выбирает задачу из выпадающего списка.

- Демо должно поддерживать загрузку произвольного изображения.

- Для каждой задачи должно быть поддержано минимум две модели на выбор. Желательно использовать модели, полученные в предыдущих частях и HW, но можно и другие.

- Демо должно отображать результаты работы модели:

Для классификации - выбранный класс и его score, runner up и его score.

Для детекции - отрисованные bounding boxes с метками классов и скорами.
Для обеих задач - время инференса модели.



In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.3 MB/s eta 0:00:00


In [ ]:
import gradio as gr
import torch
import time
import numpy as np
from PIL import Image
import torchvision.transforms as T
from transformers import AutoModelForImageClassification, AutoFeatureExtractor
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Для классификации используем модели MobileNetV2 и ViTBase. Для детекции возьмем YOLOv8n и YOLOv8l.

In [ ]:
detection_models = {"YOLOv8n": "yolov8n.pt", "YOLOv8l": "yolov8l.pt"}

classification_models = {
    "MobileNetV2 (ImageNet)": {
        "repo": "google/mobilenet_v2_1.0_224",
        "type": "hf"
    },

    "ViT Base (ImageNet)": {
        "repo": "google/vit-base-patch16-224",
        "type": "hf"
    },
}

_model_cache = {}


Напишем функции для загрузки моделей

In [ ]:
def load_detection_model(name):
    if name in _model_cache:
        return _model_cache[name]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = YOLO(detection_models[name]).to(device).eval()
    _model_cache[name] = model
    return model

def load_classification_model(name):
    if name in _model_cache:
        return _model_cache[name]

    info = classification_models[name]
    device = "cuda" if torch.cuda.is_available() else "cpu"

    repo = info["repo"]
    model = AutoModelForImageClassification.from_pretrained(repo).to(device).eval()
    extractor = AutoFeatureExtractor.from_pretrained(repo)
    _model_cache[name] = (model, extractor)
    return model, extractor

Функция для запуска детекции объектов на изображении:

In [ ]:
def run_detection(image_pil, model_name):
    if image_pil is None:
        return None, "Загрузите изображение", ""

    try:
        model = load_detection_model(model_name)
        start = time.perf_counter()
        results = model(np.array(image_pil), verbose=False)
        dt = (time.perf_counter() - start) * 1000

        annotated = results[0].plot()
        annotated_rgb = annotated[:, :, ::-1]

        boxes = []
        if len(results[0].boxes) > 0:
            cls_ids = results[0].boxes.cls.int().cpu().numpy()
            confs = results[0].boxes.conf.cpu().numpy()
            names = results[0].names
            for i, (cid, conf) in enumerate(zip(cls_ids, confs)):
                boxes.append(f"{i+1}) {names[int(cid)]}: {conf:.3f}")
        boxes_text = "\n".join(boxes) if boxes else "Объекты не найдены."

        return annotated_rgb, boxes_text, f"{dt:.1f} мс"

    except Exception as e:
        return None, f"Ошибка детекции: {e}", ""

Функция для запуска классификации изображения:

In [ ]:
def run_classification(image_pil, model_name):
    if image_pil is None:
        return "Загрузите изображение", "", ""

    try:
        model, preprocessor = load_classification_model(model_name)
        device = next(model.parameters()).device

        inputs = preprocessor(images=image_pil, return_tensors="pt").to(device)
        pixel_values = inputs["pixel_values"]

        start = time.perf_counter()
        with torch.no_grad():
              outputs = model(pixel_values)
              logits = outputs.logits
        dt = (time.perf_counter() - start) * 1000

        probs = torch.softmax(logits, dim=1)[0]
        top2_prob, top2_idx = torch.topk(probs, 2)


        id_to_label = model.config.id2label

        top1_label = id_to_label[int(top2_idx[0])]
        top1_score = float(top2_prob[0])
        runner_up_label = id_to_label[int(top2_idx[1])]
        runner_up_score = float(top2_prob[1])

        top1_text = f"{top1_label}: {top1_score:.4f}"
        runner_up_text = f"{runner_up_label}: {runner_up_score:.4f}"

        return top1_text, runner_up_text, f"{dt:.1f} мс"

    except Exception as e:
        return f"Ошибка классификации: {e}", "", ""

with gr.Blocks(title="Detection and Classification Demo") as demo:
    gr.Markdown("YOLO Детекция и Классификация")
    gr.Markdown("Выберите задачу, модель и загрузите изображение.")

    with gr.Row():
        task = gr.Radio(choices=["Detection", "Classification"], value="Detection", label="Задача")

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="Загрузите изображение")

            with gr.Group(visible=True) as detection_group:
                det_model = gr.Dropdown(choices=list(detection_models.keys()), value=list(detection_models.keys())[0], label="Модель детекции")
                det_btn = gr.Button("Запустить детекцию", variant="primary")

            with gr.Group(visible=False) as classification_group:
                cls_model = gr.Dropdown(choices=list(classification_models.keys()), value=list(classification_models.keys())[0], label="Модель классификации")
                cls_btn = gr.Button("Запустить классификацию", variant="primary")

        with gr.Column():
            det_output_img = gr.Image(label="Результат детекции", visible=True)
            det_output_text = gr.Textbox(label="Обнаруженные объекты", lines=5, visible=True)

            cls_top1 = gr.Textbox(label="Топ-1 класс", visible=False)
            cls_runner_up = gr.Textbox(label="Runner-up класс", visible=False)

            time_output = gr.Textbox(label="Время инференса", interactive=False)

    def on_task_change(selected_task):
        if selected_task == "Detection":
            return gr.update(visible=True), gr.update(visible=False), gr.update(visible=True), gr.update(visible=True), gr.update(visible=False), gr.update(visible=False)
        else:
            return gr.update(visible=False), gr.update(visible=True), gr.update(visible=False), gr.update(visible=False), gr.update(visible=True), gr.update(visible=True)

    task.change(on_task_change, inputs=task, outputs=[detection_group, classification_group, det_output_img, det_output_text, cls_top1, cls_runner_up])

    det_btn.click(fn=run_detection, inputs=[image_input, det_model], outputs=[det_output_img, det_output_text, time_output])
    cls_btn.click(fn=run_classification, inputs=[image_input, cls_model], outputs=[cls_top1, cls_runner_up, time_output])


In [ ]:
if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=7860)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9a19ab733c64223908.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
